# Python Data Cleaning

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("../Dataset/supply_chain_data.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe()

Checking Unique Values

In [ ]:
df["Product type"].unique()

In [ ]:
df["Supplier name"].unique()

In [ ]:
df["Transportation modes"].unique()

In [ ]:
df["Routes"].unique()

In [ ]:
# Rename Columns
df.columns = (
    df.columns
      .str.strip()
      .str.replace(" ", "_")
      .str.replace("-", "_")
)

In [ ]:
df.columns

In [ ]:
df.select_dtypes(include=['object']).columns

Business Features

In [ ]:
# Profit Margin
df['Profit_Margin'] = (df['Revenue_generated'] - df['Manufacturing_costs']) / df['Revenue_generated']

In [ ]:
# High / Low Revenue Category
df['Revenue_Category'] = df['Revenue_generated'].apply(
    lambda x: 'High' if x > df['Revenue_generated'].median() else 'Low'
)

In [ ]:
# Shipping Efficiency
df['Shipping_Efficiency'] = df['Shipping_times'] / df['Lead_times']

In [ ]:
df.to_csv("../Dataset/supply_chain_clean_data.csv", index=False)

# EDA

In [ ]:
df = pd.read_csv("../Dataset/supply_chain_clean_data.csv")

1. Revenue Overview

In [ ]:
# Total Revenue
df['Revenue_generated'].sum()

In [ ]:
# Average Revenue per Product
df['Revenue_generated'].mean()

In [ ]:
# Revenue Distribution
import matplotlib.pyplot as plt

plt.figure()
df['Revenue_generated'].hist()
plt.title("Revenue Distribution")
plt.xlabel("Revenue")
plt.ylabel("Frequency")
plt.show()

2. Product Analysis

In [ ]:
#Top Product Types by Revenue
df.groupby('Product_type')['Revenue_generated'].sum().sort_values(ascending=False)

In [ ]:
# Product Count
df['Product_type'].value_counts()

 3. Shipping Analysis

In [ ]:
# Average Shipping Cost by Mode
df.groupby('Transportation_modes')['Shipping_costs'].mean()

In [ ]:
# Shipping Cost Distribution
df['Shipping_costs'].hist()
plt.title("Shipping Cost Distribution")
plt.show()

4. Supplier Analysis

In [ ]:
# Top Suppliers by Revenue
df.groupby('Supplier_name')['Revenue_generated'].sum().sort_values(ascending=False)

In [ ]:
# Supplier vs Defect Rate
df.groupby('Supplier_name')['Defect_rates'].mean()

5. Inventory Analysis

In [ ]:
# Stock Levels
df['Stock_levels'].describe()

In [ ]:
# Low Stock Products
df[df['Stock_levels'] < df['Stock_levels'].median()]

6. Defect Analysis

In [ ]:
# Average Defect Rate by Product
df.groupby('Product_type')['Defect_rates'].mean()

7. Correlation Analysis

In [ ]:
import seaborn as sns

plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

Insight: Shipping costs and lead times show weak linear correlation (r = -0.12), indicating that higher shipping spend (such as Air freight at $561.71 mean route cost) is not driven by longer distance or lead time, but by suboptimal mode selection and reactive expediting.

In [ ]:
# Top 5 Products by Revenue (BEST one to add)
import matplotlib.pyplot as plt

top_products = df.groupby('Product_type')['Revenue_generated'] \
                  .sum() \
                  .sort_values(ascending=False) \
                  .head(5)

plt.figure(figsize=(8,5))
top_products.plot(kind='bar')

plt.title("Top 5 Products by Revenue")
plt.ylabel("Revenue")
plt.xlabel("Product Type")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Shipping Cost by Product Type
ship_cost = df.groupby('Product_type')['Shipping_costs'].mean()

plt.figure(figsize=(8,5))
ship_cost.plot(kind='bar')

plt.title("Average Shipping Cost by Product Type")
plt.ylabel("Shipping Cost")
plt.xlabel("Product Type")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Defect Rate by Supplier
defect_rate = df.groupby('Supplier_name')['Defect_rates'].mean().sort_values(ascending=False).head(5)

plt.figure(figsize=(8,5))
defect_rate.plot(kind='bar')

plt.title("Top Suppliers by Defect Rate")
plt.ylabel("Defect Rate")
plt.xlabel("Supplier")
plt.xticks(rotation=45)
plt.show()

# Business Insights
1. Revenue Performance
Product types show significant variation in revenue contribution, with a few categories generating the majority of total revenue.
    
2. Top Products
The top 5 products contribute a major share of revenue, indicating a strong dependency on limited high-performing products.
    
3. Supplier Performance
Supplier performance varies in terms of revenue contribution and defect rates, indicating inconsistent supplier quality.
    
4. Logistics Efficiency
Shipping costs differ across product types, suggesting variation in transportation and handling requirements.
    
5. Defect Analysis
Certain suppliers show higher defect rates, which may impact product quality and customer satisfaction.
    
6. Inventory Insight
Stock levels indicate potential overstocking in some products and low availability in others, highlighting scope for inventory optimization.
    
7. Overall Conclusion
The supply chain shows opportunities for optimization in supplier selection, inventory management, and logistics cost reduction to improve overall efficiency and profitability.



In [ ]:
df.columns

In [ ]:
# 8. Consulting Extension: ABC Segmentation, Safety Stock & Reorder Point (ROP)
import numpy as np

df = df.sort_values(by='Revenue_generated', ascending=False).reset_index(drop=True)
df['Cum_Revenue_Pct'] = df['Revenue_generated'].cumsum() / df['Revenue_generated'].sum()
df['ABC_Segment'] = np.where(
    df['Cum_Revenue_Pct'] <= 0.70, 'Class A (Top 70% Rev)',
    np.where(df['Cum_Revenue_Pct'] <= 0.90, 'Class B (Next 20% Rev)', 'Class C (Bottom 10% Rev)')
)

# Service-level Z-scores: Class A = 98% (2.05), Class B = 95% (1.65), Class C = 90% (1.28)
z_map = {'Class A (Top 70% Rev)': 2.05, 'Class B (Next 20% Rev)': 1.65, 'Class C (Bottom 10% Rev)': 1.28}
df['Daily_Demand'] = df['Number_of_products_sold'] / 30.0
df['Optimal_Safety_Stock'] = np.ceil(df['ABC_Segment'].map(z_map) * (0.25 * df['Daily_Demand']) * np.sqrt(df['Lead_times']))
df['Reorder_Point_ROP'] = np.ceil(df['Daily_Demand'] * df['Lead_times'] + df['Optimal_Safety_Stock'])
df['Inventory_Action'] = np.where(df['Stock_levels'] < df['Reorder_Point_ROP'], 'Below ROP - Trigger Order', 'Adequate Stock')

print(df.groupby('ABC_Segment')[['Revenue_generated', 'Stock_levels', 'Optimal_Safety_Stock', 'Reorder_Point_ROP']].mean().round(2))